# Pattern 2: Planner -> Executor Loop (LangGraph + Claude)

This notebook demonstrates decomposition: first create a plan, then execute each step in sequence using Claude Haiku 4.5.

```mermaid
flowchart TD
    U[Task] --> P[Planner]
    P --> E[Executor Step N]
    E --> C{More Steps?}
    C -->|Yes| E
    C -->|No| F[Done]
```

In [1]:
from typing import TypedDict, List

from langchain_anthropic import ChatAnthropic
from langgraph.graph import END, START, StateGraph

MODEL_NAME = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=MODEL_NAME, temperature=0, max_tokens=1024)

In [2]:
class PlanState(TypedDict):
    task: str
    plan_steps: List[str]
    current_step: int
    notes: List[str]

def planner_node(state: PlanState) -> PlanState:
    planning_prompt = f"""
Create a concise numbered plan (3 to 5 steps) for this task:
{state['task']}
Only return the numbered plan.
"""
    print(f"Planning for task: {state['task']}")
    print(f"Planning prompt:\n{planning_prompt}")
    plan_text = llm.invoke(planning_prompt).content
    raw_lines = [line.strip() for line in str(plan_text).splitlines() if line.strip()]
    steps = []
    for line in raw_lines:
        cleaned = line.lstrip('0123456789). -')
        if cleaned:
            steps.append(cleaned)
    if not steps:
        steps = ["Understand the problem", "Draft a solution", "Summarize clearly"]
    print(f"Generated plan steps: {steps}")
    state['plan_steps'] = steps
    state['current_step'] = 0
    state['notes'] = [f"Plan created with {len(steps)} steps."]
    return state

def executor_node(state: PlanState) -> PlanState:
    idx = state['current_step']
    step_text = state['plan_steps'][idx]
    exec_prompt = f"""
Task: {state['task']}
Current step ({idx + 1}/{len(state['plan_steps'])}): {step_text}
Give a short but useful completion for this step.
"""
    print(f"Executing step {idx + 1}: {step_text}")
    step_result = llm.invoke(exec_prompt).content
    state['notes'].append(f"Step {idx + 1}: {step_text}\n{step_result}")
    state['current_step'] = idx + 1
    return state

def route_after_executor(state: PlanState) -> str:
    if state['current_step'] >= len(state['plan_steps']):
        return 'finish'
    return 'execute'

In [3]:
graph_builder = StateGraph(PlanState)
graph_builder.add_node('planner', planner_node)
graph_builder.add_node('executor', executor_node)

graph_builder.add_edge(START, 'planner')
graph_builder.add_edge('planner', 'executor')
graph_builder.add_conditional_edges('executor', route_after_executor, {
    'execute': 'executor',
    'finish': END,
})

graph = graph_builder.compile()

initial_state = {
    'task': 'Explain how to start learning agentic AI in 2 weeks with daily actions.',
    'plan_steps': [],
    'current_step': 0,
    'notes': [],
}

final_state = graph.invoke(initial_state)
final_state['plan_steps'], final_state['notes'][-1]

Planning for task: Explain how to start learning agentic AI in 2 weeks with daily actions.
Planning prompt:

Create a concise numbered plan (3 to 5 steps) for this task:
Explain how to start learning agentic AI in 2 weeks with daily actions.
Only return the numbered plan.

Generated plan steps: ['# 2-Week Agentic AI Learning Plan', '**Days 1-3: Foundation & Concepts** - Study core agentic AI principles (agents, reasoning loops, tool use) through tutorials and documentation from OpenAI, Anthropic, or LangChain. Spend 1-2 hours daily on theory and conceptual understanding.', '**Days 4-7: Hands-On Basics** - Build your first simple agent using a framework like LangChain or AutoGPT. Follow guided tutorials to create a basic agent that can use 2-3 tools. Code for 1-2 hours daily.', '**Days 8-11: Intermediate Projects** - Develop a more complex agent with multiple tools, memory, and decision-making logic. Experiment with different prompting strategies and agent architectures. Dedicate 1.5-2 

(['# 2-Week Agentic AI Learning Plan',
  '**Days 1-3: Foundation & Concepts** - Study core agentic AI principles (agents, reasoning loops, tool use) through tutorials and documentation from OpenAI, Anthropic, or LangChain. Spend 1-2 hours daily on theory and conceptual understanding.',
  '**Days 4-7: Hands-On Basics** - Build your first simple agent using a framework like LangChain or AutoGPT. Follow guided tutorials to create a basic agent that can use 2-3 tools. Code for 1-2 hours daily.',
  '**Days 8-11: Intermediate Projects** - Develop a more complex agent with multiple tools, memory, and decision-making logic. Experiment with different prompting strategies and agent architectures. Dedicate 1.5-2 hours daily to coding and iteration.',
  '**Days 12-14: Integration & Refinement** - Build a practical agent project combining everything learned (e.g., research assistant, task automation). Test, debug, and document your work. Spend 2 hours daily finalizing and reflecting on lessons lear

## How the code cells map to the pattern
Cell 2 creates the Claude-backed planner/executor model used by both graph nodes.
Cell 3 defines the shared state plus the planner node that decomposes the task and the executor node that works one step at a time.
Cell 4 compiles the LangGraph workflow, seeds the initial state, and runs the loop until all planned steps are completed.